# BigAlpha 2026 因子提交代码 v14（提交版 · 窗口一致性根治）

## v14 相比 v13 的修复（平台判"疑似未来函数"的根因）

平台的"疑似未来函数"判定是**窗口一致性检验**：用不同 `(start_date, end_date)`
多次调用 `main()`，同一 `(date, instrument)` 的因子值必须完全一致。
本地复现实验发现 v13 有两个失配通道，v14 全部根治：

| # | 失配通道 | v13 实验证据 | v14 修复 |
|---|---|---|---|
| 1 | **rolling 热身截断**：窗口起点附近 ~60 个交易日的 rolling 特征只有截断历史 | 起点 1月→3月平移，23.9% 行不一致（max diff 2.44），集中在首月 | 查询起点前移 `WARMUP_DAYS=150` 个日历日（≈100 交易日 > 最长 60 日窗口），输出前裁回请求窗口。只多用历史数据，零未来信息 |
| 2 | **幽灵零值污染截面**：pivot 补出的格子被 `nan_to_num(0)` 填 0，当日不存在的股票以 0 值参与当日 cs_rank/cs_demean/zscore/中性化 | 仅改变股票 E 入指时间，A~D 历史因子 mean diff 0.45 | 面板保留 **NaN**；`cs_rank` 去掉 `fillna(0.5)`；`_zscore_rows` 用 NaN 初始化；缺失特征给全 NaN；幽灵格子全程不参与统计，输出时被 inner join 剔除 |
| 3 | list_days 5% 分位数阈值是全窗口统计量 | 阈值随调用窗口变化 | 改为确定性规则：非自然日量纲直接跳过新股过滤 |

## v14 回归测试结果（模拟数据，两个实验）

- **窗口平移实验**：`2023-01-01~06-30` vs `2023-03-01~06-30`，440 行重叠数据
  **max|diff| = 2.4e-15**（浮点噪声级，v13 为 2.44）——窗口平移不变 ✓
- **幽灵股票实验**：E 晚入指 vs E 永不入指，E 未入指时段 A~D 因子
  **max|diff| = 0.0**（v13 为 1.29）——幽灵格子零污染 ✓
- 输出裁剪正确（短窗口输出 min date = 请求起点）、7/7 公式 OK、factor std≈1.0

## 继承 v12/v13 的修复（未改动）

- v13：股票池 panel 模式按 `(date, instrument)` 逐日对过滤
- v12：`_resolve_tables()` 解析 dict 型 datasource；表名正则校验；主表失败回退 factorlib
- v11：分区表查询带 `filters`；`_safe_filter` 熔断；空面板熔断；date/instrument 全链路 str

## 时点安全保证（v14 加强）

- `main()` 内无训练、无标签、无随机性，只对冻结公式求值
- 时序算子全部 `shift(d)/rolling(d)`（d>0，仅后视）；截面算子仅作用当日真实存在的股票
- **窗口不变性**：热身缓冲保证 rolling 历史完整 + NaN 掩码保证截面成分正确，
  同一 `(date, instrument)` 的因子值不随调用窗口变化

## 数据源使用清单（全部为比赛指定表）

| 角色 | 默认表 | 用途 | 缺失时行为 |
|---|---|---|---|
| factorlib | `bigalpha_2026_factorlib` | 主数据源（平台 datasource 解析结果优先） | 回退默认表，仍失败才抛错 |
| instruments | `bigalpha_2026_instruments` | 股票池（逐日成分） | 跳过过滤，用主数据源全集 |
| exposure | `bigalpha_2026_exposure` | 中性化分组 | 退化为纯市值中性化 |

## 提交流程

1. 上传到 AIStudio，把最后一个 cell 的 `LOCAL_TEST` 改为 `True`
2. 跑短窗口，确认日志里 `[ds] resolved tables` 正确、`load>=` 起点前移、
   7 个公式全部 `OK`
3. 可用两个不同 start_date 各跑一次短窗口，merge 后 diff 应为 0（窗口一致性自验）
4. 改回 `LOCAL_TEST = False` 后提交

评判程序会 import 本文件的 `main` 并传入 `(datasource, start_date, end_date)`，返回值是且仅是 `[date, instrument, factor]` 三列。

In [15]:
import dai
import numpy as np
import pandas as pd
import time

# ==================== 比赛指定数据源（只可使用这些表） ====================
DS_FACTORLIB   = 'bigalpha_2026_factorlib'     # 日频因子库：close/volume/amount/turn/total_market_cap/pb/pe_ttm...
DS_INSTRUMENTS = 'bigalpha_2026_instruments'   # 股票列表（股票池）
DS_BAR1M       = 'bigalpha_2026_stock_bar1m'   # 分钟线（本版未使用）
DS_FINANCIAL   = 'bigalpha_2026_financial'     # 财务（本版未使用）
DS_EXPOSURE    = 'bigalpha_2026_exposure'      # 风险暴露（用于中性化，缺失自动跳过）

DEFAULT_START = '2019-01-01'
DEFAULT_END = '2024-12-31'

# v14: 热身缓冲（日历日）。数据查询起点比请求窗口前移这么多天，
# 使请求窗口内每个交易日的 rolling 特征（最长 60 交易日）都有完整历史，
# 平台窗口一致性检验下同一 (date, instrument) 因子值不随 start_date 变化。
# 只多用历史数据，不引入任何未来信息。150 日历日 ≈ 100 交易日 > 60 上限。
WARMUP_DAYS = 150

# ==================== 冻结公式区（v9 挖掘产出，已筛选+方向校正） ====================
# 来源：GP 挖掘（train 2019-2022），注释为该公式的样本外 IC（OOS 2023-2024）。
FROZEN_FORMULAS = [
    # F1: 换手率波动 vs 成交额（OOS_IC +0.0740）
    ('op_sub', 'turnover_std', ('op_mul', 'turnover_std', 'amount')),
    # F2: 换手率波动 - 成交额（OOS_IC +0.0753）
    ('op_sub', 'turnover_std', 'amount'),
    # F3: -(流动性偏离 - 成交额排名)（OOS_IC +0.0740）
    ('op_sub', 0.0, ('op_sub', ('cs_demean', ('op_mul', 'turnover_std', 'amount')), ('cs_rank', 'amount'))),
    # F4: 流动性偏离 + 布林位置×价值（OOS_IC +0.0738）
    ('op_add', ('cs_demean', ('op_mul', 'turnover_std', ('op_sub', 'turnover_std', 'amount'))), ('op_mul', 'bb_pct', 'bp')),
    # F5: -(开盘动量分位×成交额 + 价值量能极值)（OOS_IC +0.0750）
    ('op_sub', 0.0, ('op_add', ('cs_demean', ('op_mul', ('ts_pct', ('ts_delta', 'open', 1), 20), 'amount')), ('op_max', ('op_mul', 'bb_pct', 'bp'), 'amount'))),
    # F6: 动量/股息-流动性最小值×成交额（OOS_IC +0.0762，样本外最佳）
    ('op_mul', ('op_min', 'mo_10d', ('op_sub', 'div_yield', ('op_mul', 'turnover_std', 'amount'))), 'amount'),
    # F7: -(流动性偏离 + 动量极值组合)（OOS_IC +0.0740）
    ('op_sub', 0.0, ('op_add', ('cs_demean', ('op_mul', 'turnover_std', 'amount')), ('op_max', 'mo_10d', ('op_min', 'mo_10d', 'volume_change')))),
]

# 各公式逐日截面 zscore 后的集成权重（None = 等权）
FORMULA_WEIGHTS = None

# 是否做行业/风格中性化（逐日截面回归取残差，时点安全；数据缺失自动跳过）
NEUTRALIZE = True

# 严格模式：若任一公式所需的原始列缺失导致特征退化为常量，是否打印告警
STRICT_WARN = True

print(f"[Config] {len(FROZEN_FORMULAS)} frozen formulas, neutralize={NEUTRALIZE}, "
      f"warmup={WARMUP_DAYS}d")

[Config] 7 frozen formulas, neutralize=True, warmup=150d


## 算子库（与挖掘版完全一致，勿改动）

所有时序算子只使用历史行（rolling/shift 向后看），截面算子只作用当日。

**v14 例外**：`cs_rank` 移除了 `fillna(0.5)`，三个截面算子全部 NaN-skipna——当日不存在的股票（幽灵格子）保持 NaN、不参与当日截面统计。时序算子未改动。

In [16]:
EPS = 1e-8

# ---- Element-wise operators (7) ----
def op_add(a, b):       return a + b
def op_sub(a, b):       return a - b
def op_mul(a, b):       return a * b
def op_div(a, b):       return a / (np.abs(b) + EPS)
def op_max(a, b):       return np.maximum(a, b)
def op_min(a, b):       return np.minimum(a, b)
def op_abs(a):          return np.abs(a)

# ---- Time-series operators (10) — all backward-looking (rolling/shift use ONLY past rows) ----
def ts_delay(a, d=5):   return pd.DataFrame(a).shift(d).values
def ts_delta(a, d=5):   return a - pd.DataFrame(a).shift(d).values
def ts_mean(a, d=5):    return pd.DataFrame(a).rolling(d, min_periods=1).mean().values
def ts_std(a, d=5):     return pd.DataFrame(a).rolling(d, min_periods=1).std().fillna(0).values
def ts_rank(a, d=5):    return pd.DataFrame(a).rolling(d, min_periods=1).rank(pct=True).fillna(0.5).values
def ts_sum(a, d=5):     return pd.DataFrame(a).rolling(d, min_periods=1).sum().values
def ts_min(a, d=5):     return pd.DataFrame(a).rolling(d, min_periods=1).min().values
def ts_max(a, d=5):     return pd.DataFrame(a).rolling(d, min_periods=1).max().values
def ts_pct(a, d=5):     return pd.DataFrame(a).rolling(d, min_periods=1).rank(pct=True).fillna(0.5).values

def ts_autocorr(a, d=5):
    """Vectorized rolling autocorrelation: corr(x[t-w+1..t], x[t-w-d+1..t-d]) per column.
    Backward-looking only. w = d (window), lag = d."""
    x = pd.DataFrame(a)
    y = x.shift(d)                      # lagged series (past values only)
    mx = x.rolling(d, min_periods=3).mean()
    my = y.rolling(d, min_periods=3).mean()
    mxy = (x * y).rolling(d, min_periods=3).mean()
    mx2 = (x * x).rolling(d, min_periods=3).mean()
    my2 = (y * y).rolling(d, min_periods=3).mean()
    cov = mxy - mx * my
    vx = (mx2 - mx * mx).clip(lower=0)
    vy = (my2 - my * my).clip(lower=0)
    return (cov / (np.sqrt(vx * vy) + EPS)).fillna(0).values

# ---- Cross-sectional operators (3) — same-day cross-section only ----
def cs_rank(a):
    # v14: 不再 fillna(0.5) —— NaN 格子（当日不存在的股票）保持 NaN，
    # rank(skipna=True) 自动将其排除在当日截面之外（根治幽灵零值污染）
    df = pd.DataFrame(a)
    return df.rank(pct=True, axis=1).values

def cs_demean(a):
    return a - np.nanmean(a, axis=1, keepdims=True)

def cs_zscore(a):
    mu = np.nanmean(a, axis=1, keepdims=True)
    sigma = np.nanstd(a, axis=1, keepdims=True) + EPS
    return (a - mu) / sigma

# ---- Scalar transforms (2) ----
def np_log(a):
    return np.log(np.abs(a) + EPS)

def np_sign(a):
    return np.sign(a)

ELEMENT_OPS = ['op_add', 'op_sub', 'op_mul', 'op_div', 'op_max', 'op_min', 'op_abs']
TS_OPS = ['ts_delay', 'ts_delta', 'ts_mean', 'ts_std', 'ts_rank', 'ts_autocorr', 'ts_sum',
          'ts_min', 'ts_max', 'ts_pct']
CS_OPS = ['cs_rank', 'cs_demean', 'cs_zscore']
SCALAR_OPS = ['np_log', 'np_sign']
ALL_OPS = ELEMENT_OPS + TS_OPS + CS_OPS + SCALAR_OPS

OP_PARAMS = {
    'ts_delay': [1, 3, 5, 10, 20, 40],
    'ts_delta': [1, 3, 5, 10, 20, 40],
    'ts_mean':  [3, 5, 10, 20, 40, 60],
    'ts_std':   [5, 10, 20, 40, 60],
    'ts_rank':  [5, 10, 20, 40, 60],
    'ts_autocorr':  [5, 10, 20, 40, 60],
    'ts_sum':   [5, 10, 20, 40, 60],
    'ts_min':   [5, 10, 20, 40, 60],
    'ts_max':   [5, 10, 20, 40, 60],
    'ts_pct':   [5, 10, 20, 40, 60],
}

OP_FUNCS = {
    'op_add': op_add, 'op_sub': op_sub, 'op_mul': op_mul, 'op_div': op_div,
    'op_max': op_max, 'op_min': op_min, 'op_abs': op_abs,
    'ts_delay': ts_delay, 'ts_delta': ts_delta, 'ts_mean': ts_mean,
    'ts_std': ts_std, 'ts_rank': ts_rank, 'ts_autocorr': ts_autocorr, 'ts_sum': ts_sum,
    'ts_min': ts_min, 'ts_max': ts_max, 'ts_pct': ts_pct,
    'cs_rank': cs_rank, 'cs_demean': cs_demean, 'cs_zscore': cs_zscore,
    'np_log': np_log, 'np_sign': np_sign,
}

UNARY_OPS = ['op_abs', 'np_log', 'np_sign'] + CS_OPS

print(f"[Ops] {len(ALL_OPS)} operators loaded "
      f"({len(ELEMENT_OPS)} elem + {len(TS_OPS)} ts + {len(CS_OPS)} cs + {len(SCALAR_OPS)} scalar)")

[Ops] 22 operators loaded (7 elem + 10 ts + 3 cs + 2 scalar)


## 特征工程（v10：全列缺失容错 + 代理构造）

In [17]:
ALL_FEATURES = [
    'close', 'open', 'high', 'low', 'volume', 'turn', 'amount',
    'bp', 'ep', 'log_mktcap', 'div_yield',
    'ret_1d', 'ret_5d', 'ret_20d', 'mo_10d',
    'vol_20d', 'amihud', 'turnover_std', 'ret_vol_ratio',
    'high_low_range', 'close_open', 'close_pos', 'gap',
    'vol_ratio', 'vol_ma_ratio',
    'ret_skew', 'volume_change', 'rsi_14', 'bb_pct',
    'amount_ma_ratio',
]

# 缺列时的代理构造方案：目标列 -> (依赖列组, 构造函数说明)
_PROXY_NOTE = {
    'turn':   'amount / float_market_cap（或 volume / 流通股数）；都缺则用 volume 的相对变化代理',
    'open':   'prev_close（缺开盘价时以昨收代理，gap 恒为 0）',
    'high':   'max(close, open)',
    'low':    'min(close, open)',
}


def _ensure_columns(raw):
    """
    v10 关键修复：把所有下游用到的原始列做缺失兜底。
    比赛指定表 bigalpha_2026_factorlib 只有
      date/instrument/close/volume/amount/turn/total_market_cap/pb/pe_ttm/ps_ttm/...
    没有 open/high/low/dividend_yield_ratio/is_risk_warning，
    v9 直接裸取 raw['turn'] / raw['open'] / raw['high'] / raw['low'] 会 KeyError 崩溃。
    这里统一在特征工程之前补齐，并打印实际使用的来源，方便日志核对。
    """
    src = {}
    n = len(raw)

    # --- close 是唯一硬依赖 ---
    if 'close' not in raw.columns:
        raise KeyError("datasource must provide 'close'")

    # --- open / high / low：缺失则用 close 侧代理 ---
    if 'open' not in raw.columns:
        raw['open'] = raw.groupby('instrument')['close'].shift(1)
        raw['open'] = raw['open'].fillna(raw['close'])
        src['open'] = 'proxy: prev_close'
    else:
        src['open'] = 'raw'

    if 'high' not in raw.columns:
        raw['high'] = np.maximum(raw['close'], raw['open'])
        src['high'] = 'proxy: max(close, open)'
    else:
        src['high'] = 'raw'

    if 'low' not in raw.columns:
        raw['low'] = np.minimum(raw['close'], raw['open'])
        src['low'] = 'proxy: min(close, open)'
    else:
        src['low'] = 'raw'

    # --- volume ---
    if 'volume' not in raw.columns:
        raw['volume'] = raw['amount'] / raw['close'].clip(lower=EPS) if 'amount' in raw.columns \
            else pd.Series(1.0, index=raw.index)
        src['volume'] = 'proxy: amount / close'
    else:
        src['volume'] = 'raw'

    # --- amount ---
    if 'amount' not in raw.columns:
        raw['amount'] = raw['close'] * raw['volume']
        src['amount'] = 'proxy: close * volume'
    else:
        src['amount'] = 'raw'

    # --- turn（换手率）：6/7 个公式的核心输入，必须构造出有效值 ---
    if 'turn' in raw.columns:
        raw['turn'] = pd.to_numeric(raw['turn'], errors='coerce')
        src['turn'] = 'raw'
    elif 'float_market_cap' in raw.columns:
        raw['turn'] = raw['amount'] / raw['float_market_cap'].clip(lower=1.0) * 100.0
        src['turn'] = 'proxy: amount / float_market_cap * 100'
    elif 'total_market_cap' in raw.columns:
        raw['turn'] = raw['amount'] / raw['total_market_cap'].clip(lower=1.0) * 100.0
        src['turn'] = 'proxy: amount / total_market_cap * 100'
    else:
        # 最后退路：用成交量相对自身 60 日均值的比率作换手代理（同样只用历史数据）
        raw['turn'] = raw.groupby('instrument')['volume'].transform(
            lambda x: x / x.rolling(60, min_periods=5).mean().replace(0, np.nan))
        src['turn'] = 'proxy: volume / rolling60_mean(volume)'
    raw['turn'] = raw['turn'].replace([np.inf, -np.inf], np.nan)

    # --- 估值列 ---
    for c in ['pb', 'pe_ttm', 'total_market_cap', 'float_market_cap']:
        src[c] = 'raw' if c in raw.columns else 'missing -> fallback'

    # --- 股息率：factorlib 无此列，缺失置 0（F6 退化为 -流动性项，仍有效） ---
    if 'dividend_yield_ratio' not in raw.columns:
        raw['dividend_yield_ratio'] = 0.0
        src['dividend_yield_ratio'] = 'missing -> 0.0'
    else:
        src['dividend_yield_ratio'] = 'raw'

    print("  [cols] " + ", ".join(f"{k}={v}" for k, v in src.items()))
    return raw


def build_features(raw):
    """v11 feature engineering. All backward-looking, all columns fault-tolerant."""
    if raw is None or len(raw) == 0:
        raise RuntimeError("build_features received EMPTY raw — 上游过滤清空了数据")
    raw = raw.sort_values(['instrument', 'date']).reset_index(drop=True)
    raw = _ensure_columns(raw)
    grp = raw.groupby('instrument')

    # Backward-looking shifts (past data only)
    raw['prev_close'] = grp['close'].shift(1)
    raw['close_5d'] = grp['close'].shift(5)
    raw['close_10d'] = grp['close'].shift(10)
    raw['close_20d'] = grp['close'].shift(20)

    raw['ret_1d'] = (raw['close'] / raw['prev_close'] - 1.0)
    raw['ret_5d'] = (raw['close'] / raw['close_5d'] - 1.0)
    raw['ret_20d'] = (raw['close'] / raw['close_20d'] - 1.0)
    raw['mo_10d'] = (raw['close'] / raw['close_10d'] - 1.0)

    for c in ['prev_close', 'close_5d', 'close_10d', 'close_20d']:
        raw[c] = raw.groupby('instrument')[c].transform(lambda s: s.ffill())
        raw[c] = raw[c].fillna(raw['close'])
    for c in ['ret_1d', 'ret_5d', 'ret_20d', 'mo_10d']:
        raw[c] = raw[c].fillna(0)

    # Market cap
    if 'total_market_cap' in raw.columns:
        raw['log_mktcap'] = np.log(pd.to_numeric(raw['total_market_cap'], errors='coerce')
                                   .clip(lower=1e8).fillna(1e8) + 1)
    elif 'float_market_cap' in raw.columns:
        raw['log_mktcap'] = np.log(pd.to_numeric(raw['float_market_cap'], errors='coerce')
                                   .clip(lower=1e8).fillna(1e8) + 1)
    else:
        raw['log_mktcap'] = np.log(raw['close'] * raw['volume'].clip(lower=1) + 1)

    # Valuation
    raw['bp'] = (1.0 / pd.to_numeric(raw['pb'], errors='coerce').clip(lower=EPS)) \
        if 'pb' in raw.columns else 0.0
    raw['ep'] = (1.0 / pd.to_numeric(raw['pe_ttm'], errors='coerce').clip(lower=EPS)) \
        if 'pe_ttm' in raw.columns else 0.0
    raw['div_yield'] = pd.to_numeric(raw['dividend_yield_ratio'], errors='coerce').fillna(0)

    # Liquidity
    raw['amount'] = raw['amount'].fillna(0)
    raw['amihud'] = (np.abs(raw['ret_1d']) / (raw['amount'].clip(lower=EPS) + EPS)).fillna(0)

    # Price patterns
    raw['high_low_range'] = (raw['high'] - raw['low']) / (raw['close'].clip(lower=EPS) + EPS)
    raw['close_open'] = (raw['close'] - raw['open']) / (raw['open'].clip(lower=EPS) + EPS)
    raw['close_pos'] = ((raw['close'] - raw['low']) / (raw['high'] - raw['low'] + EPS)).fillna(0.5)
    raw['gap'] = (raw['open'] / raw['prev_close'] - 1.0).fillna(0)

    # Rolling features (group-wise, backward-looking)
    raw['vol_20d'] = grp['ret_1d'].transform(lambda x: x.rolling(20, min_periods=5).std()).fillna(0)
    raw['turnover_std'] = grp['turn'].transform(lambda x: x.rolling(10, min_periods=3).std()).fillna(0)
    raw['vol_ratio'] = grp['volume'].transform(
        lambda x: x / x.rolling(10, min_periods=3).mean().replace(0, np.nan)).fillna(1.0)
    raw['vol_ma_ratio'] = grp['volume'].transform(
        lambda x: x / x.rolling(20, min_periods=5).mean().replace(0, np.nan)).fillna(1.0)
    raw['ret_skew'] = grp['ret_1d'].transform(lambda x: x.rolling(10, min_periods=5).skew()).fillna(0)
    raw['volume_change'] = grp['volume'].transform(lambda x: x.pct_change()).fillna(0)
    raw['ret_vol_ratio'] = (raw['ret_1d'] / (raw['vol_20d'] + EPS)).fillna(0)
    raw['amount_ma_ratio'] = grp['amount'].transform(
        lambda x: x / x.rolling(20, min_periods=5).mean().replace(0, np.nan)).fillna(1.0)

    # RSI (14-day) — rolling grouped by instrument
    delta = grp['close'].diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.groupby(raw['instrument']).transform(lambda s: s.rolling(14, min_periods=5).mean())
    avg_loss = loss.groupby(raw['instrument']).transform(lambda s: s.rolling(14, min_periods=5).mean())
    raw['rsi_14'] = (100.0 - 100.0 / (1.0 + avg_gain / (avg_loss + EPS))).fillna(50.0)

    # Bollinger Band position
    ma_20 = grp['close'].transform(lambda x: x.rolling(20, min_periods=5).mean())
    std_20 = grp['close'].transform(lambda x: x.rolling(20, min_periods=5).std())
    raw['bb_pct'] = ((raw['close'] - (ma_20 - 2 * std_20)) / (4 * std_20 + EPS)).fillna(0.5)

    # Clean
    for col in ALL_FEATURES:
        if col in raw.columns:
            raw[col] = pd.to_numeric(raw[col], errors='coerce') \
                .replace([np.inf, -np.inf], np.nan).fillna(0)

    # 退化告警：某个特征全为常量说明源列缺失，公式会失效
    if STRICT_WARN:
        degraded = [c for c in ['turn', 'turnover_std', 'amount', 'bb_pct', 'bp', 'mo_10d',
                                'volume_change', 'open']
                    if c in raw.columns and float(raw[c].std() or 0.0) < 1e-12]
        if degraded:
            print(f"  [WARN] degraded (constant) features: {degraded} -> related formulas lose signal")

    return raw

In [18]:
# ==================== 公式求值器（纯函数，时点安全） ====================
def eval_formula(node, panel):
    """
    Evaluate a frozen formula tree on feature panels.
    node: feature name (str) | numeric constant | tuple tree.
    panel: dict feature -> (T, N) array.
    Returns (T, N) array (or scalar for constant-only subtrees).
    """
    if isinstance(node, str):
        if node not in panel:
            raise KeyError(f"Unknown feature in formula: {node}")
        return panel[node]
    if isinstance(node, (int, float)):
        return float(node)
    op = node[0]
    func = OP_FUNCS.get(op)
    if func is None:
        raise ValueError(f"Unknown operator: {op}")
    if op in OP_PARAMS:                       # (ts_op, child, param)
        child = eval_formula(node[1], panel)
        if np.isscalar(child):
            return child
        return func(child, node[2])
    if op in UNARY_OPS:                       # (unary_op, child)
        child = eval_formula(node[1], panel)
        if np.isscalar(child):
            return child
        return func(child)
    # binary element-wise: (op, left, right)
    left = eval_formula(node[1], panel)
    right = eval_formula(node[2], panel)
    return func(left, right)


def collect_features(node, acc=None):
    """Collect feature names referenced by formula trees."""
    if acc is None:
        acc = set()
    if isinstance(node, str):
        acc.add(node)
    elif isinstance(node, (list, tuple)):
        for part in node[1:]:
            collect_features(part, acc)
    return acc


print("[Formula] evaluator ready.")

[Formula] evaluator ready.


## 数据加载（v10：只用比赛指定表 + 运行时列名探测）

In [19]:
import json as _json
import re as _re

_TABLE_RE = _re.compile(r'^[A-Za-z_][A-Za-z0-9_.]*$')

def _valid_table(name):
    return isinstance(name, str) and bool(_TABLE_RE.match(name.strip()))


def _resolve_tables(datasource):
    """
    v12 关键修复：平台评判程序传入的 datasource 是 dict（逻辑名->真实表名映射），
    v11 直接把 dict 拼进 SQL，产生 SELECT * FROM {...}，触发
      bigdb.ParserException: Parser Error: syntax error at or near "{"
    这里统一解析为 {role: table} 映射，支持 dict / JSON字符串 / 普通表名字符串。
    """
    tables = {'factorlib': 'bigalpha_2026_factorlib',
              'instruments': 'bigalpha_2026_instruments',
              'exposure': 'bigalpha_2026_exposure',
              'financial': 'bigalpha_2026_financial',
              'bar1m': 'bigalpha_2026_stock_bar1m'}
    d = datasource
    if d is None:
        print("  [ds] datasource=None -> use default tables")
        return tables
    if isinstance(d, str):
        s = d.strip()
        if s.startswith('{'):
            try:
                d = _json.loads(s)
            except Exception:
                try:
                    import ast
                    d = ast.literal_eval(s)
                except Exception:
                    print(f"  [ds] unparsable datasource string -> defaults: {s[:80]}")
                    return tables
        elif _valid_table(s):
            tables['factorlib'] = s
            print(f"  [ds] datasource is a plain table name: {s}")
            return tables
        else:
            print(f"  [ds] invalid datasource string -> defaults: {s[:80]}")
            return tables
    if isinstance(d, dict):
        for k, v in d.items():
            v = str(v).strip()
            if not _valid_table(v):
                continue
            key = str(k).lower()
            lv = v.lower()
            if any(t in key for t in ('factorlib', 'factor', 'daily', 'price')) \
                    or 'factorlib' in lv:
                tables['factorlib'] = v
            elif any(t in key for t in ('instrument', 'universe', 'member', 'stock_list')) \
                    or 'instrument' in lv:
                tables['instruments'] = v
            elif 'exposure' in key or 'risk' in key or 'exposure' in lv:
                tables['exposure'] = v
            elif 'financial' in key or 'financial' in lv:
                tables['financial'] = v
            elif 'bar1m' in key or 'minute' in key or 'bar1m' in lv:
                tables['bar1m'] = v
    else:
        print(f"  [ds] unsupported datasource type {type(d).__name__} -> defaults")
    print("  [ds] resolved tables: " + ", ".join(f"{k}={v}" for k, v in tables.items()))
    return tables


def _apply_datasource(datasource):
    """把解析结果写回模块级 DS_* 常量，返回主数据源表名。"""
    global DS_FACTORLIB, DS_INSTRUMENTS, DS_EXPOSURE, DS_FINANCIAL, DS_BAR1M
    t = _resolve_tables(datasource)
    DS_FACTORLIB = t['factorlib']
    DS_INSTRUMENTS = t['instruments']
    DS_EXPOSURE = t['exposure']
    DS_FINANCIAL = t['financial']
    DS_BAR1M = t['bar1m']
    return t['factorlib']


def _q(sql, sdt=None, edt=None):
    """
    DAI 查询封装。v11 关键修复：
    比赛表是分区表，不带 filters 会报
      Permission Error: 请在查询表 xxx 时使用 filters 参数指定分区范围
    所以必须优先带 filters={'date': [sdt, edt]} 查询，再逐级降级。
    """
    errs = []
    if sdt is not None and edt is not None:
        for f in ({'date': [sdt, edt]}, {'date': (sdt, edt)}):
            try:
                return dai.query(sql, filters=f).df()
            except Exception as e:
                errs.append(e)
    try:
        return dai.query(sql).df()
    except Exception as e:
        errs.append(e)
        raise errs[0]


def _probe_columns(table, sdt=None, edt=None):
    """探测表的真实列名。失败返回 None（调用方走 SELECT * 兜底）。"""
    trials = []
    if sdt and edt:
        trials.append(f"SELECT * FROM {table} WHERE date >= '{sdt}' AND date <= '{edt}' LIMIT 5")
    trials.append(f"SELECT * FROM {table} LIMIT 5")
    for sql in trials:
        try:
            df = _q(sql, sdt, edt)
            if df is not None and len(df.columns) > 0:
                return list(df.columns)
        except Exception as e:
            print(f"  [probe] {table}: {str(e)[:90]}")
    return None


def _safe_filter(raw, mask, label, min_keep=0.30):
    """
    v11 关键修复：任何行过滤都不允许把数据清空。
    v10 的 list_days>=120 过滤把 118000 行全部干掉，且无人检查，
    导致空 DataFrame 一路流到 merge 才崩溃。
    这里改为：保留比例过低就整条过滤跳过，并打印告警。
    """
    n0 = len(raw)
    try:
        kept = raw[mask]
    except Exception as e:
        print(f"  [{label}] filter error, SKIPPED: {str(e)[:80]}")
        return raw
    n1 = len(kept)
    if n1 == 0:
        print(f"  [{label}] would drop ALL {n0} rows -> filter SKIPPED "
              f"(check column semantics / units!)")
        return raw
    if n1 < max(1, int(n0 * min_keep)):
        print(f"  [{label}] would keep only {n1}/{n0} ({n1/n0:.1%}) -> filter SKIPPED "
              f"(suspicious, column may not mean what we assume)")
        return raw
    print(f"  [{label}] {n0} -> {n1} rows")
    return kept.copy()


def _diag_columns(raw):
    """
    打印关键列的取值范围。用于判断比赛表给的是【原始值】还是【标准化后的因子值】。
    factorlib 若返回 z-score，则 total_market_cap 可能为负、list_days 可能很小，
    这会让若干假设（如 list_days 是自然日、mktcap 需取 log）失效。
    """
    keys = [c for c in ['close', 'open', 'volume', 'amount', 'turn',
                        'total_market_cap', 'float_market_cap',
                        'pb', 'pe_ttm', 'list_days', 'is_risk_warning']
            if c in raw.columns]
    if not keys:
        return
    print("  [diag] value ranges (判断是原始值还是标准化值):")
    for c in keys:
        v = pd.to_numeric(raw[c], errors='coerce')
        if v.notna().sum() == 0:
            print(f"    {c:20s} ALL NaN")
            continue
        print(f"    {c:20s} min={v.min():>16.4f}  med={v.median():>16.4f}  "
              f"max={v.max():>16.4f}  nan={v.isna().mean():.1%}")


def _load_universe(sdt, edt):
    """
    从比赛指定的股票池表加载股票列表。
    返回 (members DataFrame[date, instrument] 或 None, mode)
      mode='panel'  -> 逐日成分股，可做日期级 inner join
      mode='list'   -> 只有静态股票清单，只能做 instrument 级过滤
      mode='none'   -> 表不可用，不做股票池过滤（用主数据源全集）
    规则合规：不使用 cn_stock_index_component（非比赛授权表）。
    """
    cols = _probe_columns(DS_INSTRUMENTS, sdt, edt)
    if cols is None:
        print("  [universe] instruments table unavailable -> use full datasource universe")
        return None, 'none'
    print(f"  [universe] {DS_INSTRUMENTS} columns: {cols}")

    inst_col = 'instrument' if 'instrument' in cols else None
    if inst_col is None:
        for c in ['member_code', 'code', 'symbol', 'stock_code']:
            if c in cols:
                inst_col = c
                break
    if inst_col is None:
        print("  [universe] no instrument-like column -> skip universe filter")
        return None, 'none'

    has_date = 'date' in cols
    try:
        if has_date:
            sql = (f"SELECT date, {inst_col} AS instrument FROM {DS_INSTRUMENTS} "
                   f"WHERE date >= '{sdt}' AND date <= '{edt}'")
            df = _q(sql, sdt, edt)
            df['date'] = df['date'].astype(str).str[:10]
            df = df[['date', 'instrument']].dropna().drop_duplicates()
            if len(df) == 0:
                print("  [universe] empty in window -> use full datasource universe")
                return None, 'none'
            print(f"  [universe] panel mode: {df['instrument'].nunique()} stocks, {len(df)} rows")
            return df, 'panel'
        else:
            df = _q(f"SELECT {inst_col} AS instrument FROM {DS_INSTRUMENTS}", sdt, edt)
            df = df[['instrument']].dropna().drop_duplicates()
            print(f"  [universe] list mode: {len(df)} stocks")
            return df, 'list'
    except Exception as e:
        print(f"  [universe] query failed: {str(e)[:100]} -> use full datasource universe")
        return None, 'none'


def _load_daily(ds, sdt, edt):
    """从主数据源加载日频数据，列名运行时探测，只 SELECT 实际存在的列。"""
    ds = str(ds).strip()
    if not _valid_table(ds):
        raise ValueError(f"invalid table name (datasource 未正确解析?): {ds[:80]}")
    wanted = ['close', 'open', 'high', 'low', 'volume', 'turn', 'amount',
              'total_market_cap', 'float_market_cap', 'pb', 'pe_ttm', 'ps_ttm',
              'dividend_yield_ratio', 'is_risk_warning', 'list_days',
              'daily_return', 'change_ratio']
    cols = _probe_columns(ds, sdt, edt)
    if cols is None:
        print(f"  [daily] probe failed -> SELECT *")
        sql = f"SELECT * FROM {ds} WHERE date >= '{sdt}' AND date <= '{edt}'"
    else:
        print(f"  [daily] {ds} has {len(cols)} columns")
        picked = [c for c in wanted if c in cols]
        missing = [c for c in wanted if c not in cols]
        print(f"  [daily] picked: {picked}")
        if missing:
            print(f"  [daily] not in table (will be proxied/skipped): {missing}")
        sel = ['date', 'instrument'] + picked
        sql = (f"SELECT {', '.join(sel)} FROM {ds} "
               f"WHERE date >= '{sdt}' AND date <= '{edt}'")
    raw = _q(sql, sdt, edt)
    if raw is None or len(raw) == 0:
        raise RuntimeError(f"No data from {ds} in [{sdt}, {edt}]")
    # date 统一成 'YYYY-MM-DD' 字符串，避免后续 merge 出现 dtype 不一致
    raw['date'] = raw['date'].astype(str).str[:10]
    raw['instrument'] = raw['instrument'].astype(str)
    return raw


def _load_exposure(sdt, edt):
    """
    从比赛指定的风险暴露表加载中性化用的分组列。
    返回 (DataFrame[date, instrument, group] 或 None, group_col_name)
    规则合规：不使用 cn_stock_industry_component。
    """
    cols = _probe_columns(DS_EXPOSURE, sdt, edt)
    if cols is None:
        print("  [exposure] unavailable -> neutralize on size only")
        return None, None
    print(f"  [exposure] {DS_EXPOSURE} columns: {cols}")
    cand = [c for c in cols
            if any(k in c.lower() for k in ['industry', 'sector', 'sw_', 'sw2021'])]
    if not cand:
        print("  [exposure] no industry-like column -> neutralize on size only")
        return None, None
    gcol = cand[0]
    try:
        df = _q(f"SELECT date, instrument, {gcol} FROM {DS_EXPOSURE} "
                f"WHERE date >= '{sdt}' AND date <= '{edt}'", sdt, edt)
        if df is None or len(df) == 0:
            return None, None
        df['date'] = df['date'].astype(str).str[:10]
        df['instrument'] = df['instrument'].astype(str)
        print(f"  [exposure] using '{gcol}' as neutralize group, {len(df)} rows")
        return df.drop_duplicates(['date', 'instrument']), gcol
    except Exception as e:
        print(f"  [exposure] query failed: {str(e)[:100]}")
        return None, None


def load_universe_and_data(ds, sdt, edt):
    """
    加载股票池 + 日频数据 + 中性化分组。
    返回 (raw, members, umode, group_col)

    v11 相比 v10 的修复：
    1) 所有 dai.query 走 _q()，带 filters 参数（比赛表是分区表，不带会 Permission Error）
    2) 所有行过滤走 _safe_filter()，保留比例过低自动跳过（v10 被 list_days 清空全表）
    3) date/instrument 强制转 str，杜绝 merge dtype 冲突
    4) 加载后立刻打印列值域诊断，便于判断表里是原始值还是标准化值
    """
    print("[Load] Universe...")
    members, umode = _load_universe(sdt, edt)

    print("[Load] Daily data...")
    try:
        raw = _load_daily(ds, sdt, edt)
    except Exception as e:
        if str(ds) != 'bigalpha_2026_factorlib':
            print(f"  [daily] {str(ds)[:60]} failed ({str(e)[:80]}) "
                  f"-> fallback to bigalpha_2026_factorlib")
            raw = _load_daily('bigalpha_2026_factorlib', sdt, edt)
        else:
            raise
    print(f"  Rows: {len(raw)}, stocks: {raw['instrument'].nunique()}, "
          f"dates: {raw['date'].nunique()}")
    _diag_columns(raw)

    # 股票池过滤（Python 端，避免超长 IN 列表触发 SQL parser 问题）
    # v13 关键修复（成分股前视污染）：
    # v12 用全窗口 instrument 并集过滤，T 日之后才调入指数的股票会参与 T 日的
    # 截面 cs_rank/cs_demean/zscore/中性化，构成前视污染。
    # panel 模式改为按 (date, instrument) 逐日对过滤——T 日截面只含 T 日真实成分；
    # list 模式表本身无日期列，只能维持 instrument 级过滤（该局限保留）。
    if umode == 'panel' and members is not None:
        keep = set(members['date'].astype(str).str[:10] + '|'
                   + members['instrument'].astype(str))
        key = raw['date'].astype(str).str[:10] + '|' + raw['instrument'].astype(str)
        raw = _safe_filter(raw, key.isin(keep), 'universe filter (per-day)')
    elif umode == 'list' and members is not None:
        keep = set(members['instrument'].astype(str).unique())
        raw = _safe_filter(raw, raw['instrument'].isin(keep), 'universe filter')

    # ST / 风险警示过滤
    if 'is_risk_warning' in raw.columns:
        m = pd.to_numeric(raw['is_risk_warning'], errors='coerce').fillna(0) == 0
        raw = _safe_filter(raw, m, 'risk-warning filter')

    # 新股过滤：避免上市初期极端值。
    # 注意 list_days 的单位/语义在不同表里可能不同（甚至是标准化值），
    # 所以用分位数自适应阈值 + _safe_filter 双重保护。
    if 'list_days' in raw.columns:
        ld = pd.to_numeric(raw['list_days'], errors='coerce')
        if ld.notna().sum() > 0 and float(ld.max()) >= 120:
            raw = _safe_filter(raw, ld.fillna(9999) >= 120, 'list_days>=120 filter')
        elif ld.notna().sum() > 0:
            # v14: 全窗口分位数阈值随调用窗口变化，是窗口不一致的来源之一。
            # 改为确定性规则：非自然日量纲无法解释 -> 直接跳过新股过滤。
            print(f"  [list_days] max={ld.max():.4f} < 120, 该列不是自然日单位 -> "
                  f"跳过新股过滤（v14: 不再使用窗口相关的分位数阈值）")

    if len(raw) == 0:
        raise RuntimeError("Empty raw after filters — should not happen with _safe_filter.")

    # 中性化分组
    group_col = None
    if NEUTRALIZE:
        exp_df, gcol = _load_exposure(sdt, edt)
        if exp_df is not None and gcol is not None:
            raw = raw.merge(exp_df.rename(columns={gcol: '_neut_group'}),
                            on=['date', 'instrument'], how='left')
            if raw['_neut_group'].notna().sum() > 0:
                group_col = '_neut_group'
                print(f"  Neutralize group merged, coverage="
                      f"{raw['_neut_group'].notna().mean():.1%}")

    return raw, members, umode, group_col

## 主入口 main()

In [20]:
def _build_panels(raw, feat_names):
    """Pivot raw into {feature: (T, N) array} panels. Returns (panel, dates, stocks)."""
    if len(raw) == 0:
        raise RuntimeError(
            "raw is EMPTY before panel build — 上游过滤把数据清空了。"
            "检查 [universe]/[risk-warning]/[list_days] 三处过滤日志。")
    dates = sorted(raw['date'].astype(str).unique())
    stocks = sorted(raw['instrument'].astype(str).unique())
    if len(dates) == 0 or len(stocks) == 0:
        raise RuntimeError(f"empty panel axes: {len(dates)} dates x {len(stocks)} stocks")
    panel = {}
    for feat in feat_names:
        if feat in raw.columns:
            pivot = raw.pivot_table(index='date', columns='instrument',
                                    values=feat, aggfunc='first')
            pivot = pivot.reindex(index=dates, columns=stocks)
            # v14: 保留 NaN —— 当日不存在的 (date, instrument) 格子不再填 0，
            # 截面算子 skipna 后幽灵格子不参与当日统计（根治幽灵零值污染）
            panel[feat] = pivot.values.astype(float)
        else:
            print(f"  [panel] MISSING feature '{feat}' -> all-NaN (formula will be skipped)")
            panel[feat] = np.full((len(dates), len(stocks)), np.nan)
    return panel, dates, stocks


def _zscore_rows(arr):
    """Per-day cross-sectional zscore, clipped to [-3, 3]. Same-day data only.
    v14: NaN-aware —— 无效格子保持 NaN，不参与 mean/std，也不再变成 0。"""
    out = np.full(arr.shape, np.nan, dtype=float)
    for t in range(arr.shape[0]):
        row = arr[t]
        std = np.nanstd(row)
        if np.isfinite(std) and std > EPS:
            out[t] = np.clip((row - np.nanmean(row)) / std, -3, 3)
    return out


def _neutralize(factor_arr, raw, dates, stocks, group_col=None):
    """
    Per-day neutralization on log-mktcap (+ optional group dummies).
    Same-day cross-section only -> point-in-time safe.
    group_col=None 时只做市值中性化。
    """
    T, N = factor_arr.shape
    result = np.copy(factor_arr)
    use_group = group_col is not None and group_col in raw.columns

    keep = ['date', 'instrument', 'log_mktcap'] + ([group_col] if use_group else [])
    info = raw[keep].drop_duplicates(['date', 'instrument'])
    stock_idx = {s: j for j, s in enumerate(stocks)}
    by_date = {d: g for d, g in info.groupby('date')}

    for i, d in enumerate(dates):
        day = by_date.get(d)
        if day is None or len(day) < 20:
            continue
        day_stocks = day['instrument'].values
        mc = pd.to_numeric(day['log_mktcap'], errors='coerce').fillna(0.0).values
        y = np.array([factor_arr[i, stock_idx[s]] if s in stock_idx else np.nan
                      for s in day_stocks], dtype=float)

        if use_group:
            g = day[group_col].astype(str).fillna('NA').values
            cats = sorted(set(g))
            gmap = {v: k for k, v in enumerate(cats)}
            X = np.zeros((len(day_stocks), 1 + len(cats)))
            X[:, 0] = mc
            for k, v in enumerate(g):
                X[k, 1 + gmap[v]] = 1.0
        else:
            X = np.column_stack([mc, np.ones(len(day_stocks))])

        valid = np.isfinite(y) & (np.abs(y) < 1e10) & np.isfinite(X).all(axis=1)
        if valid.sum() < 20:
            continue
        try:
            beta = np.linalg.lstsq(X[valid], y[valid], rcond=None)[0]
            resid = y - X @ beta
            for k, s in enumerate(day_stocks):
                if s in stock_idx and np.isfinite(resid[k]):
                    result[i, stock_idx[s]] = resid[k]
        except Exception:
            pass
    return result


def main(datasource=None, start_date=None, end_date=None):
    """
    Submission entry point. The platform imports and calls:
        main(datasource_name, start_date, end_date)
    Returns: DataFrame with exactly [date, instrument, factor].

    POINT-IN-TIME GUARANTEE (v14 加强):
    - No training / no labels / no randomness anywhere in this file.
    - Factor value at day T depends only on data at or before T
      (backward-looking time-series ops + same-day cross-section ops).
    - Window invariance: 查询起点前移 WARMUP_DAYS 提供完整 rolling 历史，
      截面统计只含当日真实存在的股票（NaN 掩码，幽灵格子不参与），
      同一 (date, instrument) 的因子值不随调用窗口变化。

    DATA COMPLIANCE (v10):
    - Only competition-designated tables are used:
      bigalpha_2026_factorlib / bigalpha_2026_instruments / bigalpha_2026_exposure
    - datasource 参数优先；未传时默认 bigalpha_2026_factorlib
    """
    t0 = time.time()
    # v12: datasource 可能是 dict（逻辑名->真实表名），必须先解析，不能直接拼 SQL
    ds = _apply_datasource(datasource)
    sdt = str(start_date)[:10] if start_date else DEFAULT_START
    edt = str(end_date)[:10] if end_date else DEFAULT_END
    # v14 热身缓冲：数据查询起点前移 WARMUP_DAYS 个日历日，
    # 使请求窗口内每个交易日的 rolling 特征都有完整历史（窗口平移不变性），
    # 输出前再裁回 [sdt, edt]。只多用历史数据，不引入未来信息。
    load_sdt = (pd.Timestamp(sdt) - pd.Timedelta(days=WARMUP_DAYS)).strftime('%Y-%m-%d')

    print("=" * 60)
    print(f"BigAlpha 2026 factor builder v14 | {sdt} ~ {edt} | ds={ds} "
          f"| load>={load_sdt}")
    print("=" * 60)

    # ---- 1. Load ----
    needed = set()
    for _fml in FROZEN_FORMULAS:
        collect_features(_fml, needed)
    print(f"\n[1/5] Features needed by formulas: {sorted(needed)}")
    raw, members, umode, group_col = load_universe_and_data(ds, load_sdt, edt)

    # ---- 2. Feature engineering ----
    print("\n[2/5] Feature engineering...")
    raw = build_features(raw)
    print(f"  Rows: {len(raw)} | elapsed {(time.time()-t0)/60:.1f} min")

    # ---- 3. Formula evaluation ----
    print("\n[3/5] Evaluating frozen formulas...")
    panel, dates, stocks = _build_panels(raw, sorted(needed))
    print(f"  Panel shape: {len(dates)} dates x {len(stocks)} stocks")

    weights = FORMULA_WEIGHTS or [1.0] * len(FROZEN_FORMULAS)
    assert len(weights) == len(FROZEN_FORMULAS)
    wsum = float(sum(weights)) or 1.0

    factor_arr = np.zeros((len(dates), len(stocks)))
    w_used, n_ok = 0.0, 0
    for i, formula in enumerate(FROZEN_FORMULAS):
        try:
            fv = eval_formula(formula, panel)
            fv = np.asarray(fv, dtype=float)
            fv[~np.isfinite(fv)] = np.nan
            if fv.shape != factor_arr.shape:
                raise ValueError(f"shape {fv.shape} != {factor_arr.shape}")
            fv = _zscore_rows(fv)
            # v14: NaN 感知退化检测 —— 有效值比例过低或全截面无波动都判失败；
            # 幽灵格子为 NaN，不影响真实格子的判定
            _valid = np.isfinite(fv)
            if fv.size == 0 or _valid.mean() < 0.3:
                raise ValueError(f"formula output too sparse (valid={_valid.mean():.1%})")
            _sd = float(np.nanstd(fv))
            if not np.isfinite(_sd) or _sd < EPS:
                raise ValueError(f"formula output degenerate (std={_sd}) — source feature missing")
            factor_arr += (weights[i] / wsum) * fv
            w_used += weights[i] / wsum
            n_ok += 1
            print(f"  Formula #{i+1} OK (w={weights[i]/wsum:.3f})")
        except Exception as e:
            print(f"  Formula #{i+1} FAILED (skipped): {str(e)[:120]}")
    if n_ok == 0 or w_used <= 0:
        raise RuntimeError("All formulas failed to evaluate — nothing to submit.")
    if w_used < 1.0:
        factor_arr /= w_used
    print(f"  {n_ok}/{len(FROZEN_FORMULAS)} formulas evaluated.")
    if n_ok < len(FROZEN_FORMULAS):
        print(f"  [WARN] only {n_ok} formulas contributed — expected IC will be lower "
              f"than the mining-stage estimate.")

    # ---- 4. Neutralize + standardize ----
    print("\n[4/5] Neutralize + standardize...")
    if NEUTRALIZE:
        try:
            factor_arr = _neutralize(factor_arr, raw, dates, stocks, group_col)
            print(f"  Neutralization done (group={'yes' if group_col else 'size-only'}).")
        except Exception as e:
            print(f"  Neutralization skipped: {str(e)[:100]}")
    factor_arr = _zscore_rows(factor_arr)

    # ---- 5. Output: exactly 3 columns ----
    print("\n[5/5] Building output...")
    # v11: 显式指定 dtype=object，避免空列表时 numpy 推断成 float64，
    # 与 raw 的 object 型 date 列 merge 时抛
    #   ValueError: You are trying to merge on float64 and object columns
    date_arr = np.array([str(d)[:10] for d in dates], dtype=object)
    stock_arr = np.array([str(s) for s in stocks], dtype=object)
    out = pd.DataFrame({
        'date': np.repeat(date_arr, len(stocks)),
        'instrument': np.tile(stock_arr, len(dates)),
        # v14: 保留 NaN —— 幽灵格子由 real_pairs inner join 剔除，
        # 真实格子的残余 NaN 由后面的 dropna 剔除，不再静默填 0
        'factor': factor_arr.ravel(),
    })
    out['date'] = out['date'].astype(str)
    out['instrument'] = out['instrument'].astype(str)

    # 只保留主数据源里真实存在的 (date, instrument)，剔除 pivot 补出的空格子
    real_pairs = raw[['date', 'instrument']].drop_duplicates().copy()
    real_pairs['date'] = real_pairs['date'].astype(str)
    real_pairs['instrument'] = real_pairs['instrument'].astype(str)
    merged = out.merge(real_pairs, on=['date', 'instrument'], how='inner')
    if len(merged) == 0:
        raise RuntimeError(
            f"real_pairs join produced 0 rows. out={len(out)} real_pairs={len(real_pairs)}; "
            f"out.date sample={out['date'].head(3).tolist()}, "
            f"raw.date sample={real_pairs['date'].head(3).tolist()} — 日期格式不一致")
    out = merged

    # v14: 热身期数据只用于提供历史，输出严格裁回平台请求的 [sdt, edt]
    out = out[(out['date'] >= sdt) & (out['date'] <= edt)]
    if len(out) == 0:
        raise RuntimeError("Output empty after warmup trim — check sdt/edt.")

    # 若股票池是逐日面板，再按当日成分股裁剪
    if umode == 'panel' and members is not None:
        mp = members[['date', 'instrument']].drop_duplicates().copy()
        mp['date'] = mp['date'].astype(str)
        mp['instrument'] = mp['instrument'].astype(str)
        trimmed = out.merge(mp, on=['date', 'instrument'], how='inner')
        if len(trimmed) > 0:
            out = trimmed
        else:
            print("  [WARN] panel member join produced 0 rows -> keep unfiltered output")

    out = out.dropna(subset=['factor']).reset_index(drop=True)
    if len(out) == 0:
        raise RuntimeError("Output empty — check date formats / universe join.")

    out['date'] = out['date'].astype(str)
    out['instrument'] = out['instrument'].astype(str)
    out['factor'] = pd.to_numeric(out['factor'], errors='coerce').fillna(0.0).astype(float)

    print("\n" + "=" * 60)
    print(f"DONE ({(time.time()-t0)/60:.1f} min) | rows={len(out)} | "
          f"dates={out['date'].min()}~{out['date'].max()} | "
          f"stocks={out['instrument'].nunique()} | factor std={out['factor'].std():.4f}")
    print("=" * 60)
    return out[['date', 'instrument', 'factor']]


print("[main] submission entry ready.")

[main] submission entry ready.


In [21]:
# ==================== 本地自测入口（提交前必须跑通！） ====================
# 步骤：
#   1) LOCAL_TEST 改 True，跑【短窗口】（约1-2分钟）
#   2) 检查日志五处关键信息：
#      a) [probe] 行不应再出现 Permission Error（v11 已带 filters 参数）
#      b) [diag] 行看各列量纲：若 total_market_cap 中位数是 1e9~1e11 量级说明是原始值；
#         若在 -5~5 之间则该表返回的是标准化值，log_mktcap 的取 log 假设需要调整
#      c) 各过滤行的 "N0 -> N1 rows"，出现 "filter SKIPPED" 说明该列语义与假设不符（已自动兜底）
#      d) [cols] 行里 turn 是 raw 还是 proxy（若走到 volume 比率退路，IC 会明显下降）
#      e) 7 个 Formula 是否全部 OK，末行 factor std 应接近 1.0
#   3) 全部正常后改回 LOCAL_TEST = False 再提交
LOCAL_TEST = False

if LOCAL_TEST:
    # 【短窗口】快速验证全链路（先跑这个）
    #result = main(DS_FACTORLIB, '2023-01-01', '2023-06-30')
    # 【完整窗口】短窗口通过后：注释上一行，启用下一行
    result = main(DS_FACTORLIB, '2019-01-01', '2024-12-31')
    print(result.head())
    print(result.shape)
    print("dtypes:\n", result.dtypes)
    print("factor std:", result['factor'].std())
    print("nan count:", result['factor'].isna().sum())
    assert list(result.columns) == ['date', 'instrument', 'factor'], "columns mismatch!"
    print("\n>>> SELF TEST PASSED. Set LOCAL_TEST=False and submit.")


# ==================== 数据源字段自查工具（可选，排错用） ====================
# 提交前若想确认比赛表的真实字段和量纲，把 PROBE 改为 True 跑一次。
# 注意：比赛表是分区表，必须带 filters 或 WHERE date 范围，否则 Permission Error。
PROBE = False
if PROBE:
    _S, _E = '2023-01-01', '2023-01-10'
    for _t in [DS_FACTORLIB, DS_INSTRUMENTS, DS_EXPOSURE, DS_FINANCIAL]:
        _sql = f"SELECT * FROM {_t} WHERE date >= '{_S}' AND date <= '{_E}' LIMIT 5"
        try:
            _df = dai.query(_sql, filters={'date': [_S, _E]}).df()
            print(f"\n=== {_t} ({len(_df.columns)} cols) ===")
            print(list(_df.columns))
            print(_df.head(3).to_string())
            print(_df.describe().T[['min', '50%', 'max']].to_string())
        except Exception as _e1:
            try:
                _df = dai.query(_sql).df()
                print(f"\n=== {_t} (no-filters ok, {len(_df.columns)} cols) ===")
                print(list(_df.columns))
                print(_df.head(3).to_string())
            except Exception as _e2:
                print(f"\n=== {_t} FAILED ===\n  with filters: {_e1}\n  without: {_e2}")